In [ ]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoTokenizer, AutoProcessor
from qwen_vl_utils import process_vision_info

# default: Load the model on the available device(s)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2.5-VL-7B-Instruct", torch_dtype="auto", device_map="auto"
)

# We recommend enabling flash_attention_2 for better acceleration and memory saving, especially in multi-image and video scenarios.
# model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
#     "Qwen/Qwen2.5-VL-7B-Instruct",
#     torch_dtype=torch.bfloat16,
#     attn_implementation="flash_attention_2",
#     device_map="auto",
# )

# default processer
processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-7B-Instruct")

# The default range for the number of visual tokens per image in the model is 4-16384.
# You can set min_pixels and max_pixels according to your needs, such as a token range of 256-1280, to balance performance and cost.
# min_pixels = 256*28*28
# max_pixels = 1280*28*28
# processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-7B-Instruct", min_pixels=min_pixels, max_pixels=max_pixels)

# Messages containing a images list as a video and a text query
# messages = [
#     {
#         "role": "user",
#         "content": [
#             {
#                 "type": "video",
#                 "video": [
#                     "file:///path/to/frame1.jpg",
#                     "file:///path/to/frame2.jpg",
#                     "file:///path/to/frame3.jpg",
#                     "file:///path/to/frame4.jpg",
#                 ],
#             },
#             {"type": "text", "text": "Describe this video."},
#         ],
#     }
# ]

# Messages containing a local video path and a text query
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "video",
                "video": "file:///path/to/video1.mp4",
                "max_pixels": 360 * 420,
                "fps": 1.0,
            },
            {"type": "text", "text": "Describe this video."},
        ],
    }
]

# # Messages containing a video url and a text query
# messages = [
#     {
#         "role": "user",
#         "content": [
#             {
#                 "type": "video",
#                 "video": "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen2-VL/space_woaudio.mp4",
#             },
#             {"type": "text", "text": "Describe this video."},
#         ],
#     }
# ]

#In Qwen 2.5 VL, frame rate information is also input into the model to align with absolute time.
# Preparation for inference
text = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
image_inputs, video_inputs, video_kwargs = process_vision_info(messages, return_video_kwargs=True)
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    fps=fps,
    padding=True,
    return_tensors="pt",
    **video_kwargs,
)
inputs = inputs.to("cuda")

# Inference
generated_ids = model.generate(**inputs, max_new_tokens=128)
generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)
print(output_text)


# # Preparation for inference
# text = processor.apply_chat_template(
#     messages, tokenize=False, add_generation_prompt=True
# )
# image_inputs, video_inputs = process_vision_info(messages)
# inputs = processor(
#     text=[text],
#     images=image_inputs,
#     videos=video_inputs,
#     padding=True,
#     return_tensors="pt",
# )
# inputs = inputs.to("cuda")

# # Inference: Generation of the output
# generated_ids = model.generate(**inputs, max_new_tokens=128)
# generated_ids_trimmed = [
#     out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
# ]
# output_text = processor.batch_decode(
#     generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
# )
# print(output_text)


In [1]:
from typing import List, Tuple
import os

from utils.io import get_audio_binary, get_video, split_by_segments, pick_clips, merge_clips, speed_up_video
from asr import asr, SentenceSRT
from lang_clip.segmentation.full_split import text_segment_openai_call
from lang_clip.refine.selection import selection_openai_call

In [2]:
video_path = "C:\\videos\\扬哥回放\\2025-03-15 22-15-48.mp4"
output_dir = "./outputs/2025-03-15"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
sampling_rate = 16000
api_key = "sk-e03ec4e356c04d548cd1895cc3452ef7"

In [3]:
clip = get_video(video_path)

{'video_found': True, 'audio_found': True, 'metadata': {'major_brand': 'isom', 'minor_version': '512', 'compatible_brands': 'isomiso2avc1mp41', 'encoder': 'Lavf61.1.100'}, 'inputs': [{'streams': [{'input_number': 0, 'stream_number': 0, 'stream_type': 'video', 'language': None, 'default': True, 'size': [580, 1080], 'bitrate': 120000, 'fps': 60.0, 'codec_name': 'h264', 'profile': '(High)', 'metadata': {'Metadata': '', 'handler_name': 'VideoHandler', 'vendor_id': '[0][0][0][0]'}}, {'input_number': 0, 'stream_number': 1, 'stream_type': 'audio', 'language': None, 'default': True, 'fps': 48000, 'bitrate': 96, 'metadata': {'Metadata': '', 'handler_name': 'SoundHandler', 'vendor_id': '[0][0][0][0]'}}], 'input_number': 0}], 'duration': 5748.0, 'bitrate': 120111, 'start': 0.0, 'default_video_input_number': 0, 'default_video_stream_number': 0, 'video_codec_name': 'h264', 'video_profile': '(High)', 'video_size': [580, 1080], 'video_bitrate': 120000, 'video_fps': 60.0, 'default_audio_input_number':

In [ ]:
wav = get_audio_binary(clip, sr=sampling_rate)
sentences:List[SentenceSRT] = asr(wav)

In [5]:
import os
import logging
import json

from llm.openai_api import openai_call

text_seg_system_prompt = """你是女装直播视频剪辑师。请分析以下按时间顺序排列的字幕句子列表，精确识别所有产品介绍段落，并且剪辑出精彩片段。
# 方法介绍
1. 精准识别出不同商品的介绍部分（先分段，再总结，再提取关键词）
2. 提取出每个商品的介绍片段，保留最精彩的部分。要求：卖点+品牌+衣服尺码(不需要价格)
3. 把最精彩的部分放在开头

# 注意事项
1. **过渡关键词**：使用明确结束当前产品的词汇（如"过掉了""换款""下一个"）；分界后的句子会描述新品属性（颜色/尺码/材质/价格）;还有一些特殊的引入词，比如“准备一下”，“给大家上一个”“再给你们上”表示介绍新品
2. **卖点关键词**：卖点是指商品的特色描述，吸引人的点，如"新款"、"超级好搭"、"超级显瘦"等。适用场景、适用人群、适用季节等也是卖点的一部分。
3. **品牌关键词**：“xx”家的“xx”款，或者“xx”品牌的“xx”款
3. **排除干扰**：忽略重复性口语填充词（如"嗯""啊"）和未切换产品的数量说明（如"只剩3件"）。过度词汇也要剔除。和女装无关的内容也要剔除，例如玩笑什么的。
5. **尺码关键词**：码，长度，胸围，可以穿到xx斤
4. **长度要求**：十句以内即可，保留精彩部分。

# 输出格式
- array of json objects: [{"indices": [10, 11, 13, 15], "卖点": "春夏季新款","重点":[11, 13]}, ...,] （不需要markdown format）


# 示例分析
输入：
20. 你说不了话
21. 菜菜很难受
22. 再给大家上一个哦
23. 很有设计感的一件真的衬衫
24. 这个是夏天穿的一个衬衫
25. 很有调性是siammisr家的一个新款
26. 这个牌子的话称之为日本的小松本全t
27. 这个是均码长度七十二斤
28. 胸围一百零六
29. 可以穿到一百二十五斤

输出：[{"indices":[23, 24, 25, 26, 27, 28, 29], "卖点": "有设计感、夏天的衬衫","重点":[24, 25]}, ...] 
解释：句子20是口语过渡，不是分界点；21是无关话题；句子22是明确的切换过渡，后续开始介绍新品。24, 25是最精彩部分，放在开头。"""


text_seg_user_prompt_template = """请分析以下直播字幕的分界点
输入：
{}

输出：
"""


def text_segment_openai_call(
    apikey, 
    字幕列表, 
    model="deepseek-reasoner",
    system_prompt=text_seg_system_prompt,
    user_content=None,
):
    if not user_content:
        subtitle_list = "\n".join([f"{i}. {text}" for i, text in enumerate(字幕列表)])
        user_content = text_seg_user_prompt_template.format(subtitle_list)
    logging.info("Openai model inference done.")
    output = openai_call(apikey, model, user_content, system_prompt, is_json=False)
    logging.info(output)
    print(output)
    json_output = json.loads(output)
    return json_output

In [6]:
boundaries = text_segment_openai_call(
    apikey=api_key,
    model='deepseek-reasoner',
    字幕列表=[ele.text for ele in sentences],
)

[
  {"indices": [18, 19, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 36, 37, 38, 39, 40, 41, 42, 45, 46, 47, 48, 49, 50, 52, 55, 56, 57, 58, 59, 60, 71, 72, 73, 74, 76, 77, 78, 79], "卖点": "日本高端品牌英伯格连衣裙 均码可穿130斤 注重面料品质", "重点": [24, 29, 31]},
  {"indices": [83, 84, 85, 86, 90, 91, 92, 93, 94, 95, 96], "卖点": "nanouniverse春款休闲半身裙 粗花呢面料 S/M码可选", "重点": [84, 85]},
  {"indices": [116, 117, 118, 119, 120, 121, 122, 124, 125, 126, 127], "卖点": "lowpickany碎花半身裙 三色可选 腰围权限大", "重点": [116, 121]},
  {"indices": [204, 205, 206, 208, 209, 210, 211, 212, 213, 214], "卖点": "cocodell短款风衣 小个子专属 A字版四季可穿", "重点": [204, 206]},
  {"indices": [238, 239, 240, 242, 243, 244, 245, 246, 247, 248, 252, 268, 269], "卖点": "flyID抽绳设计纱衫 杂志同款 双层防透工艺", "重点": [238, 245]},
  {"indices": [277, 278, 279, 280, 281, 282, 283, 284, 285, 291, 292], "卖点": "salfor质感纱裙 杂志款 腰围精准裁剪", "重点": [277, 280]},
  {"indices": [314, 315, 316, 317, 318, 319, 320], "卖点": "rinda鱼尾半裙 绑带设计 内衬工艺优秀", "重点": [314, 316]},
  {"indices": [362, 363, 364, 366,

In [ ]:
from utils.io import pick_clips, merge_clips
import os
import re

def sanitize_filename(filename):
    # 定义不同操作系统中的非法字符（示例为通用规则）
    illegal_chars = r'[\\/:\*\?"<>\|]'  # 匹配 \ / : * ? " < > | 等
    return re.sub(illegal_chars, '_', filename) 

for ele in boundaries:
    indices = ele['重点'] + ele['indices']
    clips = pick_clips(clip, sentences, indices)
    merge_clip = merge_clips(clips)
    name = sanitize_filename(ele['卖点'])
    merge_clip[0].write_videofile(os.path.join(output_dir, f"{name}.mp4"), codec="libx264", audio_codec="aac")

In [8]:
merge_clip[1]

[SentenceSRT(text='这个是micamicadue的一个新款', _duration=Duration(start=Timestamp(hour=0, minute=0, second=0, millisecond=0), end=Timestamp(hour=0, minute=0, second=3, millisecond=100)), token_timestamps=[Duration(start=Timestamp(hour=0, minute=0, second=0, millisecond=0), end=Timestamp(hour=0, minute=0, second=0, millisecond=80)), Duration(start=Timestamp(hour=0, minute=0, second=0, millisecond=80), end=Timestamp(hour=0, minute=0, second=0, millisecond=180)), Duration(start=Timestamp(hour=0, minute=0, second=0, millisecond=180), end=Timestamp(hour=0, minute=0, second=0, millisecond=300)), Duration(start=Timestamp(hour=0, minute=0, second=0, millisecond=300), end=Timestamp(hour=0, minute=0, second=0, millisecond=840)), Duration(start=Timestamp(hour=0, minute=0, second=1, millisecond=580), end=Timestamp(hour=0, minute=0, second=2, millisecond=60)), Duration(start=Timestamp(hour=0, minute=0, second=2, millisecond=60), end=Timestamp(hour=0, minute=0, second=2, millisecond=280)), Duration(start=

In [2]:
funasr_model = AutoModel(
    model="iic/speech_seaco_paraformer_large_asr_nat-zh-cn-16k-common-vocab8404-pytorch",
    vad_model="damo/speech_fsmn_vad_zh-cn-16k-common-pytorch",
    punc_model="damo/punc_ct-transformer_zh-cn-common-vocab272727-pytorch",
    spk_model="damo/speech_campplus_sv_zh-cn_16k-common",
    disable_update=True,
    device='cuda:0'
)

funasr version: 1.2.6.


2025-03-23 18:07:17,819 - modelscope - WARNING - Using branch: master as version is unstable, use with caution


2025-03-23 18:07:20,392 - modelscope - WARNING - Using branch: master as version is unstable, use with caution


2025-03-23 18:07:21,313 - modelscope - WARNING - Using branch: master as version is unstable, use with caution


2025-03-23 18:07:22,790 - modelscope - WARNING - Using branch: master as version is unstable, use with caution


Detect model requirements, begin to install it: C:\Users\szzso\.cache\modelscope\hub\models\damo\speech_campplus_sv_zh-cn_16k-common\requirements.txt
install model requirements successfully


In [3]:
video_path = "C:\\videos\\扬哥回放\\2025-03-15 21-35-19.mp4"
output_dir = "./outputs"
sampling_rate = 16000


clip = VideoFileClip(video_path)

audio_path = join(output_dir, "audio.wav")
try:
    clip.audio.write_audiofile(audio_path)
    wav, sampling_rate = librosa.load(audio_path, sr=sampling_rate)
finally:
    if exists(audio_path):
        remove(audio_path)
wav = convert_pcm_to_float(wav)

{'video_found': True, 'audio_found': True, 'metadata': {'major_brand': 'isom', 'minor_version': '512', 'compatible_brands': 'isomiso2avc1mp41', 'encoder': 'Lavf61.1.100'}, 'inputs': [{'streams': [{'input_number': 0, 'stream_number': 0, 'stream_type': 'video', 'language': None, 'default': True, 'size': [580, 1080], 'bitrate': 120000, 'fps': 60.0, 'codec_name': 'h264', 'profile': '(High)', 'metadata': {'Metadata': '', 'handler_name': 'VideoHandler', 'vendor_id': '[0][0][0][0]'}}, {'input_number': 0, 'stream_number': 1, 'stream_type': 'audio', 'language': None, 'default': True, 'fps': 48000, 'bitrate': 136, 'metadata': {'Metadata': '', 'handler_name': 'SoundHandler', 'vendor_id': '[0][0][0][0]'}}], 'input_number': 0}], 'duration': 2427.88, 'bitrate': 120151, 'start': 0.0, 'default_video_input_number': 0, 'default_video_stream_number': 0, 'video_codec_name': 'h264', 'video_profile': '(High)', 'video_size': [580, 1080], 'video_bitrate': 120000, 'video_fps': 60.0, 'default_audio_input_number

MoviePy - Done.


In [10]:
rec_result = funasr_model.generate(
    wav, 
    return_spk_res=True, 
    sentence_timestamp=True, 
    return_raw_text=True, 
    # is_final=True, 
    # hotword="姐妹",
    # output_dir=output_dir,
    # cache={}
)

rtf_avg: -3.188: 100%|██████████| 1/1 [00:03<00:00,  3.19s/it]
rtf_avg: 0.011, time_speech:  2427.880, time_escape: 27.805: 100%|██████████| 1/1 [00:27<00:00, 27.81s/it]


In [11]:
rec_result[0]['sentence_info'][0]

{'text': '家的一个新款半裙哦，',
 'start': 110,
 'end': 1630,
 'timestamp': [[110, 210],
  [210, 310],
  [310, 390],
  [390, 530],
  [530, 750],
  [750, 990],
  [990, 1150],
  [1150, 1390],
  [1390, 1630]],
 'raw_text': '家的一个新款半裙哦',
 'spk': 0}

In [5]:
from funclip.utils.subtitle_utils import time_convert
text_clips = [(ele['text'], ele['start'], ele['end'], time_convert(ele['start']), time_convert(ele['end'])) for ele in rec_result[0]['sentence_info']]

In [6]:
import gc
import torch

clip.close()
del funasr_model, wav, rec_result
torch.cuda.empty_cache()  # Clears the cache of CUDA memory
gc.collect()  # Call garbage collector

1675

In [20]:
# Use a pipeline as a high-level helper
from transformers import pipeline
import torch
pipe = pipeline("text-generation", model="Qwen/Qwen2.5-7B-Instruct", device=0, torch_dtype=torch.bfloat16)

Loading checkpoint shards: 100%|██████████| 4/4 [00:00<00:00,  6.16it/s]
Device set to use cuda:0


In [13]:


system_content = (
    "你是一个女装直播字幕分段助手。请根据当前句子和上下文，判断当前句子是否为「产品介绍段落的分界点」。规则如下：\n"
    "1. 分界点通常是主播结束当前产品、切换新产品的过渡句（如“过掉了”“再上一个”）。\n"
    "2. 分界句后的内容会开始描述新产品的特征（如颜色、材质、价格）。\n"
    "3. 注意排除重复口语（如“这个过掉了，过掉了哈”可能只有第二句是分界点）。\n"
    "输入格式：\n"
    "- 当前句子：[句子序号] [句子内容]\n"
    "- 上下文（最近几句）：[句子内容...]\n"
    "请用 JSON 格式回答：\n"
    "{\n"
    "  \"is_boundary\": true/false,\n"
    "  \"reason\": \"分界理由或排除理由\"\n"
    "}\n"
    "示例输入1：\n"
    "当前句子：过掉了哈"
    "上下文：够这个不多了，\n这个过掉了，\n过掉了哈，\n然后再给大家上一个刚出完货的，"
    "示例输出1："
    "{\n"
    "  \"is_boundary\": true,\n"
    "  \"reason\": \"主播明确用‘过掉了哈’结束当前产品，且后文开始新产品的介绍（如‘再给大家上一个’）\"\n"
    "}\n"
    "示例输入2：\n"
    "当前句子：只有 s 码了啊，"
    "上下文：嗯，啊，只有 s 码了啊，少女的妈妈也可以，这这裙子真的很优雅，就你感觉你要给自己一个机会去感受一下。"
    "示例输出2："
    "{\n"
    "  \"is_boundary\": false,\n"
    "  \"reason\": \"这只是推销的话术\"\n"
    "}\n"
)

user_prompt = (
    "当前句子：它够五件了，"
    "上下文：少女的妈妈也可以，这这裙子真的很优雅，就就你感觉你要给自己一个机会去感受一下。你其实其实你可以这么优雅，你不要觉得说我平时你像我我平时穿的就是那样子，但是我也可以很酷，你知道吗？就是你可以感受一下，因为这种裙子它很优雅，它够五件了，哥感一一够五件，这个不多了，这个掉掉了，掉掉了，这个过掉了，过掉了哈，然后再给大家上一个刚出完货的，"
)

messages = [
    {"role": "system", "content": system_content},
    {"role": "user", "content": user_prompt},
]



In [8]:
import os
import logging
import json

from openai import OpenAI

system_prompt = """你是女装直播字幕分段专家。请分析以下按时间顺序排列的字幕句子列表，精确识别所有产品介绍段落的分界点。

# 分界点特征
1. **过渡标志**：使用明确结束当前产品的词汇（如"过掉了""换款""下一个"）
2. **新品引入**：分界后的句子会描述新品属性（颜色/尺码/材质/价格）;
还有一些特殊的引入词，比如“再准备一下”，“给大家上一个”“再给你们上”表示介绍新品
3. **上下文特征**：分界点通常出现在连续短句的最后一个（如"这个过掉了，过掉了哈"中后者为分界）。临近的不要作为分界点。上下文有关联的不要作为分界点。
4. **排除干扰**：忽略重复性口语填充词（如"嗯""啊"）和未切换产品的数量说明（如"只剩3件"）

# 输出要求
- 直接返回可供json.load的字符串：{"boundaries": [分界句序号列表]}。省略Markdown的```json```的格式。
- 序号从0开始，仅包含明确符合上述条件的句子编号

# 示例分析
输入：
0. 够这个不多了，
1. 这个过掉了，
2. 过掉了哈，
3. 然后再给大家上一个刚出完货的

输出：{"boundaries": [2]} 
解释：句子1是口语过渡，不是分界点；句子2是明确的切换过渡，后续开始介绍新品。"""


user_prompt_template = """请分析以下直播字幕的分界点
输入：
{}

输出（按JSON格式返回结果）：
"""


def text_segment_openai_call(
    apikey, 
    字幕列表, 
    model="deepseek-reasoner",
    base_url="https://api.deepseek.com/v1",
    system_content=system_prompt,
    user_content=None,
):
    client = OpenAI(
        # This is the default and can be omitted
        api_key=apikey,
        base_url=base_url
    )
    subtitle_list = "\n".join([f"{i}. {text}" for i, text in enumerate(字幕列表)])
    messages = [
        {'role': 'system', 'content': system_content},
        {'role': 'user', 'content': user_content if user_content else user_prompt_template.format(subtitle_list)}
    ]
    
    chat_completion = client.chat.completions.create(
        messages=messages,
        model=model,
    )
    logging.info("Openai model inference done.")
    output = chat_completion.choices[0].message.content
    print(output)
    json_output = json.loads(output)
    return json_output['boundaries']
    

In [ ]:
import json
output = pipe(messages, max_new_tokens=500)[0]['generated_text'][-1]['content']
json.loads(output)['is_boundary']

In [11]:
first_elements_iterator = (t[0][:-1] for t in text_clips)
print("\n".join([f"{i}. {text}" for i, text in enumerate(first_elements_iterator)]))

# print(text_clips[:20])

0. 家的一个新款半裙哦
1. 它这个的话边上有点小开叉设计
2.  s 码长度九十
3. 腰围六十八 m 码
4. 长度九十四
5. 腰围七十四 s 码
6. 穿到一百零五斤 m 码
7. 穿到一百一十五斤
8. 一百二十九一号链接上车
9. 你这帅卖
10. 我老婆已经被我拉黑了
11. 这是 u 型开叉
12.  u 型开叉姐妹跟平时那种开叉不一样
13. 它是 u 型开叉
14. 这个对对
15. 师傅的要求很高的
16. 对尔 simiss 的小松本的短裙没有了
17. 七宝七宝没有关注的帮我关注点一下吧
18. 给你们上一个春天必备的一个很薄的一个小小西装外套吧
19. 没有了
20. 这个也是我们刚刚出完货
21. 这个西装很薄
22. 对
23. 你老婆也因为退货多吗
24. 哦
25. 是我老婆
26. 因为在直播间老是要乱讲话
27. 这个很薄哈
28. 春夏
29. 这个是我们给日本品牌啊
30. 来了
31. 有点难来了
32. 卡皮瑞 copy CV lameaga 啊
33. 这个牌子对一句日文不太会读
34. 但他们家很棒
35. 他们家衣服在日本是一个小众的高端品牌
36. 然后是一个薄款
37. 嗯
38. 才两万两万两万日元
39. 姐妹日本
40. 然后这款的话是均码长度七十二
41. 胸围一百一十六
42. 穿到一百三十斤
43. 日本三月二十号才开始预售的款式
44. 对你今天在我这里下单全部是晒仓
45. 你比日本人优先拿到这件衣服
46.  unbelievable
47. 如果被客人知道的话
48. 要死的一百一十斤
49. 你穿你小个子就拍 s 码
50. 你个子高就拍 m 码
51. 陈小艺
52. 没想到我认识那个字吧
53. 很有文化
54. 不是很这个穿到一百三十斤
55. 一百七十九一号链接上车
56. 你是帅帅是开头
57. 遇到那件吗
58. 不是
59. 这是另外一个款
60. 好
61. 格子可以给你们试啊
62. 没有关注的新粉
63. 记得关注点一下
64. 格子要试一下吗
65. 你先把卡其色拍完嘛
66. 先在不说
67. 我不说话
68. 你走上格子
69. 给他们套一下
70. 特别好好格子
71. 可以给你们试了
72. 姐妹嘛
73. 对嘛
74. 这就对了嘛
75. 格子就六件了


In [ ]:
first_elements_iterator = (t[0][:-1] for t in text_clips)
response = text_segment_openai_call("sk-e03ec4e356c04d548cd1895cc3452ef7", first_elements_iterator, model='deepseek-reasoner')
response

# {"boundaries": [17, 101, 157, 226, 259, 395, 494, 530, 694, 773, 795, 859, 1123]}

{"boundaries": [18, 158, 227, 260, 396, 400, 495, 547, 567, 860, 1124]}


[18, 158, 227, 260, 396, 400, 495, 547, 567, 860, 1124]

In [ ]:
outputs = [637]
for idx in outputs:
    print(text_clips[idx-1])
    print(text_clips[idx])
    print(text_clips[idx+1])


# [46, 102, 303, 359, 402, 562, 637, 708, 849]
# 401. 因为下一句是：给大家上一个。。。
# 637. 再给你们上一个

{"boundaries": [45, 101, 359, 402, 562, 708, 848]}

('但是不能被抓抓了的话，', 1173910, 1175270, '00:19:33,910', '00:19:35,270')
('会被罚二十块，', 1175270, 1176145, '00:19:35,270', '00:19:36,145')
('再给你们上一个你去做一个平地，', 1177150, 1180810, '00:19:37,150', '00:19:40,810')


In [6]:
from tqdm import trange

slide_window_size = 9
half_window_size = slide_window_size // 2
skip_until = 0

bounds = []

for i in trange(len(text_clips)):
    if i < half_window_size:
        continue
    if i >= len(text_clips) - half_window_size:
        break
    if i < skip_until:
        continue
    user_prompt = (
        f"当前句子：{text_clips[i][0]}\n"
        f"上下文：{''.join([ele[0] for ele in text_clips[i-slide_window_size:i+half_window_size+1]])}\n"
    )
    messages = [
        {"role": "system", "content": system_content},
        {"role": "user", "content": user_prompt},
    ]
    output = openai_call("sk-e03ec4e356c04d548cd1895cc3452ef7", 'deepseek-reasoner', user_prompt, system_content)
    is_boundary = json.loads(output)['is_boundary']
    if is_boundary:
        bounds.append(i)
        skip_until = i + half_window_size 
    

 24%|██▎       | 4/17 [00:00<?, ?it/s]


NameError: name 'system_content' is not defined

In [49]:
i = 10
j = bounds[i]
user_prompt = (
    f"当前句子：{text_clips[j][0]}\n"
    f"上下文：{''.join([ele[0] for ele in text_clips[j-slide_window_size:j+half_window_size+1]])}\n"
)
print(user_prompt)

当前句子：它够五件了，
上下文：少女的妈妈也可以，这这裙子真的很优雅，就就你感觉你要给自己一个机会去感受一下。你其实其实你可以这么优雅，你不要觉得说我平时你像我我平时穿的就是那样子，但是我也可以很酷，你知道吗？就是你可以感受一下，因为这种裙子它很优雅，它够五件了，哥感一一够五件，这个不多了，这个掉掉了，掉掉了，



In [51]:
len(bounds)

161

In [36]:
text_clips[j+half_window_size]

('就这么直接。', 78340, 79100, '00:01:18,340', '00:01:19,100')